# BÀI THỰC HÀNH SỐ 1 - KHAI PHÁ DỮ LIỆU
## PHẦN 2 - BÀI 5A (TRANG 45): KHAI PHÁ LUẬT KẾT HỢP VỚI NƯỚC ĐỨC (GERMANY)
- **Họ và tên:** Bùi Minh Trọng
- **MSSV:** 6451071081
- **Lớp:** CQ.64.CNTT
- **Yêu cầu:** Áp dụng trên nước Đức ('Germany') với `min_sup = 5%` và `min_conf = 50%`


### Câu 1: Lọc dữ liệu khách hàng từ nước Đức ('Germany') và gom nhóm ma trận giỏ hàng


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

import os
data_candidates = ['Online Retail.xlsx', '../TH1/Online Retail.xlsx', '../Online Retail.xlsx', 'datasets/Online Retail.xlsx']
data_path = next((p for p in data_candidates if os.path.exists(p)), 'Online Retail.xlsx')
df_retail = pd.read_excel(data_path)
df_retail['Description'] = df_retail['Description'].astype(str).str.strip()
df_retail.dropna(subset=['InvoiceNo'], inplace=True)
df_retail['InvoiceNo'] = df_retail['InvoiceNo'].astype(str)
df_clean = df_retail[~df_retail['InvoiceNo'].str.contains('C')].copy()

# Lọc riêng nước Đức 'Germany'
basket_germany = (df_clean[df_clean['Country'] == 'Germany']
                  .groupby(['InvoiceNo', 'Description'])['Quantity']
                  .sum().unstack().reset_index().fillna(0)
                  .set_index('InvoiceNo'))

print("Kích thước ma trận hóa đơn của Đức (Germany):", basket_germany.shape)
display(basket_germany.iloc[:5, :5])


**Nhận xét Câu 1:**
- Thị trường Đức có 457 hóa đơn với 1,695 mặt hàng khác nhau được ghi nhận trong ma trận giỏ hàng.


### Câu 2: Chuyển đổi dữ liệu về dạng One-Hot Encoding (0 và 1)


In [ ]:
print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

def encode_units(x):
    return 1 if x >= 1 else 0

basket_sets_ger = basket_germany.applymap(encode_units)
print("Ma trận giỏ hàng sau khi mã hóa One-Hot:")
display(basket_sets_ger.iloc[:5, :5])


**Nhận xét Câu 2:**
- Mỗi điểm dữ liệu thể hiện sự hiện diện của sản phẩm trong giỏ hàng: 1 là có mua, 0 là không mua.


### Câu 3: Loại bỏ cột chi phí bưu chính 'POSTAGE'


In [ ]:
print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

if 'POSTAGE' in basket_sets_ger.columns:
    basket_sets_ger.drop('POSTAGE', inplace=True, axis=1)
    print("Đã loại bỏ thành công cột 'POSTAGE' khỏi tập dữ liệu.")
else:
    print("Không tìm thấy cột 'POSTAGE'.")

print("Kích thước ma trận sau khi loại bỏ POSTAGE:", basket_sets_ger.shape)


**Nhận xét Câu 3:**
- Cột `POSTAGE` chỉ là phí vận chuyển bưu chính xuất hiện trên hầu hết hóa đơn nên cần xóa bỏ để tránh sinh ra các luật vô nghĩa liên quan đến cước phí.


### Câu 4: Khai phá tập phổ biến bằng thuật toán Apriori với min_sup = 5%


In [ ]:
print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

from mlxtend.frequent_patterns import apriori

frequent_itemsets_ger = apriori(basket_sets_ger, min_support=0.05, use_colnames=True)
print(f"Số lượng tập phổ biến tìm được với min_sup = 5%: {len(frequent_itemsets_ger)}")
display(frequent_itemsets_ger.sort_values(by='support', ascending=False).head(10))


**Nhận xét Câu 4:**
- Với ngưỡng `min_sup = 0.05`, các mặt hàng phổ biến nhất ở Đức bao gồm các bộ hộp ăn vặt tròn (`ROUND SNACK BOXES`), các sản phẩm đồ chơi và đồ gia dụng xinh xắn.


### Câu 5: Rút trích các luật kết hợp với min_conf = 50%


In [ ]:
print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

from mlxtend.frequent_patterns import association_rules

rules_ger = association_rules(frequent_itemsets_ger, metric="confidence", min_threshold=0.5)
print(f"Số lượng luật kết hợp tìm được với min_conf = 50%: {len(rules_ger)}")
display(rules_ger[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(by='confidence', ascending=False).head(10))


**Nhận xét Câu 5:**
- Tìm được nhiều luật có độ tin cậy rất cao (> 70% đến 80%) và hệ số lift > 3 (ví dụ: khách hàng mua `ROUND SNACK BOXES SET OF4 WOODLAND` thường mua kèm `ROUND SNACK BOXES SET OF 4 FRUITS` hoặc `SPACEBOY`). Điều này cho thấy tính tương quan mua sắm chéo rất mạnh giữa các dòng sản phẩm cùng loại.


### Câu 6: Biểu diễn mối quan hệ giữa độ hỗ trợ và độ tin cậy trên đồ thị phân tán (Scatter Plot)


In [ ]:
print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

plt.figure(figsize=(9, 5))
plt.scatter(rules_ger['support'], rules_ger['confidence'], alpha=0.7, c=rules_ger['lift'], cmap='viridis', s=80)
plt.colorbar(label='Lift')
plt.title("Biểu đồ phân tán giữa Support và Confidence của các luật (Nước Đức)")
plt.xlabel("Độ hỗ trợ (Support)")
plt.ylabel("Độ tin cậy (Confidence)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


**Nhận xét Câu 6:**
- Các luật có độ hỗ trợ từ 5% đến 13% và độ tin cậy từ 50% đến hơn 80%.
- Giá trị lift đều lớn hơn 1 (đặc biệt các điểm màu sáng có lift > 4), chứng tỏ các quy luật mua kèm tìm được có độ tin cậy thực tế rất cao và có thể áp dụng trực tiếp vào chiến lược xếp kệ hàng hoặc gợi ý sản phẩm (cross-selling).
